In [ ]:
import pandas as pd

# 定义参数
chunk_size = 100000  # 每个块的大小
data_file = "data/characteristics_with_ind_interaction_return.pkl"

# 读取整个数据文件以获取总行数
data_sample = pd.read_pickle(data_file)
total_rows = len(data_sample)

# 逐块读取和保存
for i, offset in enumerate(range(0, total_rows, chunk_size)):
    chunk = data_sample.iloc[offset:offset + chunk_size]
    chunk.to_pickle(f"data/chunk_{i}.pkl")

print("Data has been split into smaller chunks.")

In [ ]:
import gc
import pandas as pd
import numpy as np
import pickle
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import r2_score
import os

# 加载数据
with open("data/characteristics_with_ind_interaction_return.pkl", "rb") as f:
    data = pickle.load(f)

# 查看内存使用情况
memory_info = os.popen('free -h').read()
print(memory_info)

def train_pls(data):
    # 60年数据
    start_year_train = 1957
    start_year_test = 2006
    end_year_test = 2016
    
    # 将 'DATE' 列转换为日期格式（如果尚未转换）
    data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')
    
    # 用于累积所有年份的预测和真实值
    overall_pred_y_list = []
    overall_true_y_list = []
    
    # 循环处理每一年
    for year in tqdm(range(start_year_test, end_year_test+1), desc='Processing', unit='year'):
        # 定义训练和测试的起止日期
        start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
        end_date_train = pd.Timestamp(year=year-13, month=12, day=31)
    
        start_date_val = pd.Timestamp(year=year-12, month=1, day=1)
        end_date_val = pd.Timestamp(year=year-1, month=12, day=31)
        
        start_date_test = pd.Timestamp(year=year, month=1, day=1)
        end_date_test = pd.Timestamp(year=year, month=12, day=31)
        
        print(f'训练开始日期: {start_date_train.strftime("%Y-%m")}')
        print(f'训练结束日期: {end_date_train.strftime("%Y-%m")}')
        print(f'验证开始日期: {start_date_val.strftime("%Y-%m")}')
        print(f'验证结束日期: {end_date_val.strftime("%Y-%m")}')
        print(f'测试开始日期: {start_date_test.strftime("%Y-%m")}')
        print(f'测试结束日期: {end_date_test.strftime("%Y-%m")}')
    
        # 选择训练和测试数据
        train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_val)]
        test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
        
        # 提取特征和目标变量
        X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values  
        y_train = train_data['RET'].values  
        X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
        y_test = test_data['RET'].values
        
        # 标准化特征
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
        
        # 训练PLS模型
        pls_model = PLSRegression(n_components=10)  # 选择适当的n_components
        pls_model.fit(X_train, y_train)
       
        # 使用PLS进行预测
        y_test_pred_pls = pls_model.predict(X_test).ravel()
    
        # 保存预测结果到CSV文件
        results_df = pd.DataFrame({
            'Date': test_data['DATE'],
            'True_RET': y_test,
            'Pred_RET': y_test_pred_pls
        })
        results_df.to_csv(f'pls_results/{year}.csv', index=False)
        
        # 累积所有年份的预测和真实值
        overall_pred_y_list.extend(y_test_pred_pls)
        overall_true_y_list.extend(y_test)
        
        # 输出每年的 R² 分数
        print(f'PLS R^2 for {year}: {r2_score(y_test, y_test_pred_pls) * 100}')
        
        # 清理内存
        del train_data, test_data, X_train, y_train, X_test, y_test, y_test_pred_pls, results_df
        gc.collect()
    
    # 计算整体 R² 分数
    overall_r2 = r2_score(np.array(overall_true_y_list), np.array(overall_pred_y_list)) * 100
    print('Overall R^2: {}'.format(overall_r2))

train_pls(data)